# Class-weighted baseline comparison

This experiment changes only logistic-regression class weighting. Features, split, scaling, seed, regularization, and iteration limit remain fixed.
The original run remains the reference. The balanced run uses a separate output directory.
Training for the recorded comparison was explicitly authorized by Timo.
Default execution of this notebook reads saved results and creates plots only.
Use notebook 05 for environment setup and data preparation on a fresh checkout.


In [ ]:
import json
import logging
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import Image, display

from sleep_classifier.experiment_report import ExperimentComparisonReport

logging.basicConfig(
    level=logging.INFO,
    stream=sys.stdout,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
    force=True,
)
logger = logging.getLogger("weighted_comparison_notebook")
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/baseline_balanced.yaml").exists()
)
RUN_TRAINING = False

## Optional: repeat the weighted fit

Run this cell only after choosing to repeat the experiment. It overwrites artifacts/baseline-balanced.
Preserve previous results or choose another configured output directory first.
The report runner skips this training cell regardless of the switch.
The class weights are computed from training labels. Validation labels do not select the weights.


In [ ]:
if RUN_TRAINING:
    %cd "{ROOT}"
    !uv run --locked --group notebook --group baseline python -u -m sleep_classifier.baseline \
        --config configs/baseline_balanced.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Weighted baseline fit failed")
else:
    logger.info("Training skipped; using saved comparison results")

## Export reference predictions without fitting

This command uses the original saved scaler and classifiers. It does not refit either model.
The prediction CSV supports chronological plots of both models on the same retained validation epochs.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group notebook --group baseline python -m sleep_classifier.baseline \
    --config configs/baseline.yaml \
    --export-predictions artifacts/baseline/validation_predictions.csv
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Saved-model prediction export failed")

## Compare saved metrics

Reports check the split, feature schema, source provenance, class supports, and coverage counts.
Plots generated here stay under artifacts. Selected measured figures are versioned under docs/experiments.
The final test participants remain reserved.


In [ ]:
output = ROOT / "artifacts/comparisons/class-weighted-baseline"
summary_path = ExperimentComparisonReport(
    ROOT / "artifacts/baseline/metrics.json",
    ROOT / "artifacts/baseline-balanced/metrics.json",
    output,
).run()
summary = json.loads(summary_path.read_text())
for figure in summary["figures"].values():
    if figure:
        display(Image(filename=str(output / figure)))

## Interpretation

Confusion rows represent true stages; columns represent predicted stages. Diagonal cells show correct predictions.
Recall measures the fraction of each true stage detected. F1 also accounts for precision.
Balanced accuracy is mean recall over supported classes. It differs from overall accuracy under class imbalance.
Coefficients describe standardized logit associations. Correlated inputs limit their interpretation as individual feature importance.
Timeline gaps represent missing or rejected epochs, not continuous predictions.

The recorded weighted model raises REM recall but lowers accuracy and macro-F1. It is not the overall winner.
Keep the unweighted model as the baseline reference before testing the causal GRU.


## HTML report

```bash
uv run --locked --group notebook --group baseline python scripts/execute_notebook.py --notebook notebooks/06_class_weighted_comparison.ipynb
```

The runner saves artifacts/notebooks/06_class_weighted_comparison.executed.html.
It does not execute the tagged training cell.
